In [1]:
import sys
print(sys.executable)

/opt/homebrew/Cellar/jupyterlab/4.4.2_1/libexec/bin/python


In [2]:
!which python
!which pip

/Users/wolfidy7/miniforge3/bin/python
/Users/wolfidy7/miniforge3/bin/pip


In [3]:
import sys
!{sys.executable} -m pip install pandas pillow imagehash tqdm matplotlib seaborn opencv-python beautifulsoup4

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from bs4 import BeautifulSoup
import warnings
import os
import cv2
import hashlib
import imagehash

from PIL import Image
from tqdm import tqdm
from collections import Counter

warnings.filterwarnings('ignore')

In [5]:
# **CONFIGURATION**


In [6]:
IMAGES_PATH = "image_train"
VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

In [7]:
# **RECUPRATION DES IMAGES**

In [8]:
# Nombre max d'images à charger
MAX_IMAGES = None  # mettre None pour tout charger

image_paths = []

files = os.listdir(IMAGES_PATH)

for file in files:

    if file.lower().endswith(VALID_EXTENSIONS):

        full_path = os.path.join(IMAGES_PATH, file)

        # Vérifie que c'est bien un fichier
        if os.path.isfile(full_path):

            image_paths.append(full_path)

            # Stop si limite atteinte
            if MAX_IMAGES is not None and len(image_paths) >= MAX_IMAGES:
                break

print(f"Nombre d'images trouvées : {len(image_paths)}")

Nombre d'images trouvées : 84916


In [9]:
# **STRUCTURES DE STOCKAGE**

In [10]:
results = []

exact_hashes = {}
perceptual_hashes = {}

duplicate_exact = []
duplicate_visual = []

color_stats = []

corrupted_images = []
blank_images = []
grayscale_images = []

In [11]:
#**Fonctions utilitaires**

In [12]:
#**Hash exact (fichiers identiques)**
def compute_md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

In [13]:
#**Détection image vide / noire**

In [14]:
def is_blank_image(img, threshold=5):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return np.std(gray) < threshold

In [15]:
#**Détection grayscale**
def is_grayscale(img):
    if len(img.shape) < 3:
        return True

    b, g, r = cv2.split(img)

    return (np.allclose(r, g, atol=2) and np.allclose(g, b, atol=2))

In [16]:
# **Analyse** **principale**

In [17]:
for path in tqdm(image_paths):

    try:
        # Chargement image
        img_cv = cv2.imread(path)

        if img_cv is None:
            corrupted_images.append(path)
            continue

        height, width = img_cv.shape[:2]
        ratio = width / height

        # Noir/blanc
        blank = is_blank_image(img_cv)

        if blank:
            blank_images.append(path)

        # grayscale
        gray = is_grayscale(img_cv)

        if gray:
            grayscale_images.append(path)

        # Couleurs moyennes
        mean_b = np.mean(img_cv[:, :, 0])
        mean_g = np.mean(img_cv[:, :, 1])
        mean_r = np.mean(img_cv[:, :, 2])

        color_stats.append([mean_r, mean_g, mean_b])

        # Hash exact
        md5 = compute_md5(path)

        if md5 in exact_hashes:
            duplicate_exact.append((path, exact_hashes[md5]))
        else:
            exact_hashes[md5] = path

        # Hash perceptuel (doublons visuels)
        pil_img = Image.open(path)

        phash = str(imagehash.phash(pil_img))

        if phash in perceptual_hashes:
            duplicate_visual.append((path, perceptual_hashes[phash]))
        else:
            perceptual_hashes[phash] = path

        # Stockage stats
        results.append({
            "path": path,
            "width": width,
            "height": height,
            "ratio": ratio,
            "blank": blank,
            "grayscale": gray
        })

    except Exception as e:
        corrupted_images.append(path)

100%|███████████████████████████████████████| 84916/84916 [08:14<00:00, 171.82it/s]


In [18]:
# **DataFrame**
df = pd.DataFrame(results)

print(df.head())

                                                path  width  height  ratio  \
0  image_train/image_1174594490_product_294064072...    500     500    1.0   
1  image_train/image_1172460449_product_185143847...    500     500    1.0   
2  image_train/image_1313553701_product_419925281...    500     500    1.0   
3  image_train/image_1008107210_product_435919430...    500     500    1.0   
4  image_train/image_1190251689_product_273864457...    500     500    1.0   

   blank  grayscale  
0  False      False  
1  False      False  
2  False      False  
3  False      False  
4  False      False  


In [19]:
# Résumé global

In [20]:

def print_image_list(label, paths, max_display=None):
    print(f"\n--- {label} ({len(paths)}) ---")
    items = paths if max_display is None else paths[:max_display]
    for p in items:
        print(f"  {os.path.basename(p)}")
    if max_display and len(paths) > max_display:
        print(f"  ... et {len(paths) - max_display} autres")

print("===== DATASET SUMMARY =====")
print(f"Total images analysées : {len(df)}")
print(f"Images corrompues      : {len(corrupted_images)}")
print(f"Images vides/noires    : {len(blank_images)}")
print(f"Images grayscale       : {len(grayscale_images)}")
print(f"Duplicates exacts      : {len(duplicate_exact)}")
print(f"Duplicates visuels     : {len(duplicate_visual)}")

if corrupted_images:
    print_image_list("Images corrompues", corrupted_images)

if blank_images:
    print_image_list("Images vides/noires", blank_images)

if grayscale_images:
    print_image_list("Images grayscale", grayscale_images)

if duplicate_exact:
    print(f"\n--- Duplicates exacts ({len(duplicate_exact)}) ---")
    for path, original in duplicate_exact:
        print(f"  {os.path.basename(path)}  <->  {os.path.basename(original)}")

if duplicate_visual:
    print(f"\n--- Duplicates visuels ({len(duplicate_visual)}) ---")
    for path, original in duplicate_visual:
        print(f"  {os.path.basename(path)}  <->  {os.path.basename(original)}")


===== DATASET SUMMARY =====
Total images analysées : 84916
Images corrompues      : 0
Images vides/noires    : 50
Images grayscale       : 961
Duplicates exacts      : 5692
Duplicates visuels     : 7590

--- Images vides/noires (50) ---
  image_970432492_product_259364744.jpg
  image_1198890740_product_2310115709.jpg
  image_1032419657_product_541874293.jpg
  image_1264908122_product_3928057747.jpg
  image_1267306230_product_3942385086.jpg
  image_854978064_product_87886098.jpg
  image_1137819811_product_1892606336.jpg
  image_941635457_product_207159381.jpg
  image_922216635_product_170942527.jpg
  image_1310336620_product_4182236046.jpg
  image_1174588122_product_2940639046.jpg
  image_1249535030_product_3819980401.jpg
  image_1142089742_product_884747735.jpg
  image_1252146776_product_3838117313.jpg
  image_1323073274_product_4231410877.jpg
  image_1283678514_product_4061597712.jpg
  image_1174591359_product_2940640054.jpg
  image_1307766815_product_4172823378.jpg
  image_933425072_